# AMS-02 ECAL Calorimetry and Geometry

## Block 0 physics notebook

This notebook records the calorimetry and detector-geometry foundations used by the AMS-02 ECAL simulation. The detector-level classification task is

$$e^\pm \text{ versus } p,$$

because ECAL shower morphology can separate electromagnetic from many hadronic events, while the silicon tracker supplies charge-sign and momentum information.

Statements are labeled throughout as **verified detector facts**, **derived quantities**, **modeling assumptions**, or **validation targets**. This prevents a simplified FastMC choice from being mistaken for a documented AMS-02 property.

Block 0 defines geometry only. It does not yet generate showers, project tracks, digitize signals, or create ML inputs. Detector constants live in `configs/geometry.yaml`; reusable behavior lives in `src/ams_ecal/geometry.py`.

## 1. What a calorimeter measures

A calorimeter estimates incident energy by allowing a particle to interact with matter and measuring part of the resulting energy deposition. A useful bookkeeping relation is

$$E_0 = E_{\mathrm{active}} + E_{\mathrm{passive}} + E_{\mathrm{leakage}} + E_{\mathrm{invisible}}.$$

The active scintillating fibers convert some deposited energy into light. Energy deposited in lead is not observed directly; leakage escapes the instrument; and some processes do not become measured signal. Calibration maps the visible response to a reconstructed energy $\hat E$.

A segmented calorimeter also preserves spatial information. It records where energy is deposited longitudinally and laterally, which makes shower shape useful for particle identification.

> **Verified detector fact:** AMS-02 ECAL is a lead–scintillating-fiber sampling calorimeter.

> **Validation target:** a later detector-response model should reproduce documented energy response and resolution, not merely conserve a chosen toy signal.

## 2. Electromagnetic shower formation

At high energy, electrons and positrons lose energy mainly through bremsstrahlung in the electric field of a nucleus:

$$e^\pm + Z \rightarrow e^\pm + \gamma + Z.$$

A sufficiently energetic photon can convert into an electron–positron pair:

$$\gamma + Z \rightarrow e^- + e^+ + Z.$$

Repeating these processes creates an electromagnetic cascade. The number of shower particles initially grows while their typical energy decreases. Once secondary-electron energies approach the critical-energy scale, ionization and excitation dominate, multiplication ends, and the shower decays.

The longitudinal profile therefore has three qualitative regions:

1. **Growth:** bremsstrahlung and pair production multiply particles.
2. **Shower maximum:** the charged-particle population is largest.
3. **Decay:** ionization dominates and the remaining energy is deposited.

Electrons and positrons have essentially the same ECAL shower physics. ECAL alone therefore does not provide the sign of their electric charge.

## 3. Why lead and scintillating fibers are combined

Lead is a dense passive absorber that encourages electromagnetic shower development. Scintillating fibers are active material: charged shower particles excite the scintillator, which emits light that is transported to photodetectors.

The measurement chain is approximately

$$\text{incident particle} \rightarrow \text{shower} \rightarrow \text{active deposition} \rightarrow \text{light} \rightarrow \text{electrical signal} \rightarrow \hat E.$$

Because only alternating samples of the shower are active, identical incident energies do not yield identical signals. Shower development, sampling, photon production and transport, photodetector statistics, calibration, leakage, and electronic noise all contribute fluctuations.

A standard resolution parameterization is

$$\frac{\sigma_E}{E}=\sqrt{\left(\frac{a}{\sqrt{E}}\right)^2+b^2+\left(\frac{c}{E}\right)^2},$$

where $a$ is the stochastic term, $b$ the constant term, and $c$ the noise term. The AMS overview reports approximately

$$\frac{\sigma_E}{E}=\sqrt{\frac{0.104^2}{E}+0.014^2},$$

with $E$ in GeV. This is a **validation target** for a later response model, not a Block 0 geometry rule.

In [1]:
from math import sqrt

def reported_relative_resolution(energy_gev: float) -> float:
    if energy_gev <= 0:
        raise ValueError("energy_gev must be positive")
    return sqrt(0.104**2 / energy_gev + 0.014**2)

for energy_gev in (1.0, 10.0, 100.0, 1000.0):
    print(f"{energy_gev:6.1f} GeV: {reported_relative_resolution(energy_gev):.3%}")

   1.0 GeV: 10.494%
  10.0 GeV: 3.574%
 100.0 GeV: 1.744%
1000.0 GeV: 1.438%


## 4. Natural material scales

### 4.1 Radiation length $X_0$

Radiation length is the natural longitudinal scale of a high-energy electromagnetic shower. It characterizes bremsstrahlung energy loss by electrons and pair production by photons. Physical millimetres locate detector hardware; depth in $X_0$ describes shower evolution across materials. AMS-02 ECAL has a documented total depth of about $17X_0$.

### 4.2 Critical energy $E_c$

The critical energy is the approximate scale where radiative and collisional losses become comparable for electrons in a material. It is material dependent. Higher-energy primaries produce a deeper shower maximum, approximately logarithmically rather than linearly:

$$t_{\max} \sim \ln(E_0/E_c)+C,$$

where $t_{\max}$ is measured in radiation lengths and the constant depends on the particle and profile convention. We will not assign an effective $E_c$ to the lead–fiber mixture until the longitudinal model is introduced and sourced.

### 4.3 Molière radius $R_M$

The Molière radius characterizes lateral electromagnetic-shower spread. Roughly 90% of the shower energy is contained within a cylinder of radius $R_M$ under the usual definition; containment near $2R_M$ is greater but should not be treated as a detector-independent exact percentage.

This is the physical motivation for a local crop around the tracker-projected shower axis. The proposed 21-cell width remains an experimental preprocessing choice that must be checked through containment and crop-width ablations.

## 5. Why proton events differ

Protons interact mainly through hadronic processes, whose relevant longitudinal scale is the nuclear interaction length $\lambda_I$. AMS-02 ECAL is deep electromagnetically ($17X_0$) but only about $0.6\lambda_I$ hadronically.

Consequently, a proton may traverse much of the calorimeter before a strong interaction, deposit energy like a penetrating charged particle, or initiate a partially contained and irregular hadronic cascade. A hadronic interaction can also create electromagnetic subshowers, so no single shape rule perfectly separates the classes.

Useful discriminating structure can include:

- early versus late deposition;
- longitudinal compatibility with an electromagnetic profile;
- lateral compactness;
- shower-axis consistency;
- separated or irregular deposits;
- leakage and penetrating activity;
- ECAL energy relative to tracker momentum, $E/p$.

A credible proton generator is much harder than an electromagnetic parameterization. Any FastMC proton model must be labeled as approximate and later compared with Geant4 or another suitable reference.

## 6. Documented AMS-02 ECAL construction

The official AMS-02 reconstruction description reports the following high-level structure:

| Quantity | Value | Evidence class |
|---|---:|---|
| Active transverse area | $648\times648\ \mathrm{mm}^2$ | Verified detector fact |
| Active depth | $166.5\ \mathrm{mm}$ | Verified detector fact |
| Electromagnetic depth | $17X_0$ | Verified detector fact |
| Hadronic depth | about $0.6\lambda_I$ | Verified detector fact |
| Superlayers | 9 | Verified detector fact |
| Superlayer thickness | $18.5\ \mathrm{mm}$ | Verified detector fact |
| Readout layers | 18 (two per superlayer) | Verified detector fact |
| Cells per readout layer | 72 | Verified detector fact |
| Total cells | 1296 | Verified and independently derived |
| PMTs | 324 with four anodes each | Verified detector fact |
| Anode active area | $9\times9\ \mathrm{mm}^2$ | Verified detector fact |
| Fibers | about 50,000, diameter $1\ \mathrm{mm}$ | Verified detector fact |

Successive superlayers alternate fiber directions: five have fibers parallel to one transverse axis and four to the other. Exact software names for the *fiber direction* and the *coordinate measured by a layer* will be defined separately in Block 2 to avoid an axis-convention bug.

In [2]:
from pathlib import Path

from ams_ecal import load_geometry

#Load the geometry configuration from the repository root. 
def find_repository_root(start: Path) -> Path:
    """Find the repository root from Jupyter's current directory."""

    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate

    raise FileNotFoundError("Could not locate the repository root.")


repository_root = find_repository_root(Path.cwd())
geometry = load_geometry(repository_root / "configs" / "geometry.yaml")

# Print the geometry configuration values.
(
    geometry.cell_pitch_mm,
    geometry.mean_readout_slice_thickness_mm,
    geometry.total_cells,
)

(9.0, 9.25, 1296)

The $9\ \mathrm{mm}$ transverse pitch agrees with the documented anode scale and with $648/72$. The value $166.5/18=9.25\ \mathrm{mm}$ is only a **mean readout-slice thickness**. Treating the 18 layer reference depths as uniformly spaced centers is a **Block 0 modeling assumption**, not a claim that every internal material component is a homogeneous $9.25\ \mathrm{mm}$ slab.

## 7. The event representation

The natural raw readout abstraction is

$$\mathbf E_{\mathrm{raw}}=(E_{\ell,c})\in\mathbb R_{\ge 0}^{18\times72},$$

where $\ell$ indexes longitudinal readout layer and $c$ indexes one of 72 cells in that layer. It is not a dense $18\times72\times72$ voxel cube: each row is one transverse view, with complementary views supplied by alternating superlayers.

Later, a tracker state will be projected to every layer. Each layer will select the projected coordinate it measures, locate the corresponding central cell, and extract 10 neighboring cells on each side:

$$\mathbf E_{\mathrm{crop}}\in\mathbb R_{\ge 0}^{18\times21}.$$

This is a layer-wise projected strip, not a square image crop. Boundary behavior must be explicit: a crop near an edge will need padding or a documented event-selection rule.

## 8. Coordinate and geometry contract for Block 0

The current configuration defines:

- origin at the center of the ECAL front face;
- $+z$ pointing into the ECAL;
- $x,y\in[-324,324]\ \mathrm{mm}$ for the active transverse extent;
- $z\in[0,166.5]\ \mathrm{mm}$ for the simplified active depth;
- polar angle $\theta$ measured from $+z$.

Block 0 code should load and validate configuration, expose bounds and derived counts, and produce simplified reference depths for the 18 readout layers. It should reject malformed, inconsistent, non-finite, or non-positive geometry.

The code should **not** yet decide alternating readout orientation, map coordinates to cell indices, project tracks, simulate energy deposition, or apply the $18\times21$ crop. Those belong to later blocks.

In [3]:
#Validate the geometry 
assert geometry.number_of_superlayers == 9
assert geometry.number_of_layers == 18
assert geometry.cells_per_layer == 72
assert geometry.total_cells == 1296

assert geometry.cell_pitch_mm == 9.0
assert geometry.mean_readout_slice_thickness_mm == 9.25

assert geometry.x_bounds_mm == (-324.0, 324.0)
assert geometry.y_bounds_mm == (-324.0, 324.0)
assert geometry.z_bounds_mm == (0.0, 166.5)

assert len(geometry.uniform_layer_centers_z_mm) == 18
assert geometry.uniform_layer_centers_z_mm[0] == 4.625
assert geometry.uniform_layer_centers_z_mm[-1] == 161.875

geometry.uniform_layer_centers_z_mm

(4.625,
 13.875,
 23.125,
 32.375,
 41.625,
 50.875,
 60.125,
 69.375,
 78.625,
 87.875,
 97.125,
 106.375,
 115.625,
 124.875,
 134.125,
 143.375,
 152.625,
 161.875)

## 9. Scientific-status ledger

| Item | Status in this project |
|---|---|
| Active dimensions, 9 superlayers, 18 layers, 72 cells/layer, $17X_0$ | Verified detector facts |
| $9\ \mathrm{mm}$ pitch, 1296 cells, symmetric transverse bounds | Derived consistency checks |
| Origin at front-face center and $+z$ into ECAL | Explicit software coordinate convention from configuration |
| Uniformly spaced readout-layer reference centers | Simplified Block 0 modeling assumption |
| Exact effective $E_c$ and $R_M$ of the composite calorimeter | Not yet selected or sourced |
| Reported energy-resolution curve | Later detector-response validation target |
| 21-cell tracker-centered crop | Planned preprocessing hypothesis requiring ablation |
| Electromagnetic and proton shower generators | Outside Block 0 |


## 10. References

1. AMS Collaboration, [The Electromagnetic Calorimeter (ECAL)](https://ams02.space/detector/electromagnetic-calorimeter-ecal).
2. AMS Collaboration, [New Reconstruction Method in the Electromagnetic Calorimeter (ECAL) Analysis](https://ams02.space/advances-data-analysis/new-reconstruction-method-electromagnetic-calorimeter-ecal-analysis).
3. AMS Collaboration, [Improvements in the proton rejection using ECAL](https://ams02.space/advances-data-analysis/improvements-proton-rejection-using-ecal).
4. Particle Data Group, [Passage of Particles Through Matter](https://pdg.lbl.gov/).

The AMS pages support the detector construction and reported performance targets. The PDG review supplies the general shower-physics definitions. Later blocks should cite the exact source used for every numerical parameter introduced into a physics model.